# 🎭 Détection & Génération d'Émotions par Contexte Visuel de Scène
## Pipeline YOLO + Architecture Multi-Branches sur EMOTIC

**Temps total estimé : ~8-12h GPU sur T4 (Colab Pro)**

| Section | Description | Temps |
|---------|-------------|-------|
| 0 | Installation & Setup | ~2 min |
| 1 | Téléchargement EMOTIC | manuel |
| 2 | Préparation des données | ~2 min |
| 3 | Extraction features YOLO (Composant 1) | ~10 min |
| 4 | Entraînement du détecteur (Composant 2) | ~2-3h |
| 5 | Évaluation test set | ~5 min |
| 6 | Fine-tuning LoRA SD (Composant 3) | ~4-6h |
| 7 | Génération & évaluation inter-tâches | ~30 min |
| 8 | Visualisations poster | ~1 min |

---
## Section 0 — Installation & Setup

In [ ]:
!pip install -q ultralytics albumentations scipy torchmetrics clean-fid
!pip install -q diffusers transformers peft accelerate
!pip install -q scikit-learn pandas matplotlib seaborn

In [ ]:
import os
import sys
import time
import json
import csv
import random
from collections import defaultdict
from dataclasses import dataclass, field
from typing import List, Tuple

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy.io import loadmat
from scipy.stats import pearsonr
from sklearn.metrics import average_precision_score

print(f"PyTorch : {torch.__version__}")
print(f"GPU disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  Device : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM   : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} Go")

---
## Configuration centralisée

Tous les hyperparamètres, chemins et constantes du projet. **Modifiez les chemins ici** si votre structure est différente.

In [ ]:
"""
Configuration centrale du projet EMOTIC — Détection & Génération d'Émotions Contextuelles.
Tous les hyperparamètres, chemins et constantes sont centralisés ici.
"""

import os
from dataclasses import dataclass, field
from typing import List, Tuple

# ─────────────────────────────────────────────
# Catégories EMOTIC (26 classes discrètes)
# ─────────────────────────────────────────────
EMOTIC_CATEGORIES = [
    'Affection', 'Anger', 'Annoyance', 'Anticipation', 'Aversion',
    'Confidence', 'Disapproval', 'Disconnection', 'Disquietment', 'Doubt/Confusion',
    'Embarrassment', 'Engagement', 'Esteem', 'Excitement', 'Fatigue',
    'Fear', 'Happiness', 'Pain', 'Peace', 'Pleasure',
    'Sadness', 'Sensitivity', 'Suffering', 'Surprise', 'Sympathy', 'Yearning'
]

NUM_CATEGORIES = len(EMOTIC_CATEGORIES)  # 26
NUM_VAD = 3  # Valence, Arousal, Dominance

# Classes COCO (80) — sorties YOLOv8
NUM_COCO_CLASSES = 80

# ─────────────────────────────────────────────
# VAD descriptors pour le conditioning génératif
# ─────────────────────────────────────────────
VAD_DESCRIPTORS = {
    'valence': {
        'high': ['joyful', 'pleasant', 'happy', 'delightful'],
        'mid': ['neutral', 'calm', 'composed'],
        'low': ['sad', 'unpleasant', 'gloomy', 'melancholic']
    },
    'arousal': {
        'high': ['energetic', 'intense', 'exciting', 'dynamic'],
        'mid': ['moderate', 'steady'],
        'low': ['calm', 'serene', 'tranquil', 'peaceful']
    },
    'dominance': {
        'high': ['empowered', 'confident', 'in control'],
        'mid': ['balanced', 'neutral'],
        'low': ['submissive', 'overwhelmed', 'vulnerable']
    }
}


@dataclass
class PathConfig:
    """Chemins du projet — adapter selon Colab ou Kaggle."""
    # Racine du projet
    project_root: str = '/content/emotic_project'

    # Dataset EMOTIC
    emotic_root: str = '/content/datasets/emotic'
    emotic_annotations: str = '/content/datasets/emotic/Annotations'
    emotic_images_root: str = '/content/datasets/emotic'  # contient mscoco/, emotic_subfolders/

    # Fichiers d'annotations pré-processés (générés par prepare_data.py)
    emotic_mat_file: str = '/content/datasets/emotic/Annotations/Annotations.mat'

    # Pré-computed YOLO features
    yolo_features_dir: str = '/content/emotic_project/yolo_features'

    # Checkpoints
    checkpoint_dir: str = '/content/emotic_project/checkpoints'

    # Outputs de génération
    generation_output_dir: str = '/content/emotic_project/generated'

    # Logs
    log_dir: str = '/content/emotic_project/logs'

    def makedirs(self):
        for path in [self.yolo_features_dir, self.checkpoint_dir,
                     self.generation_output_dir, self.log_dir]:
            os.makedirs(path, exist_ok=True)


@dataclass
class YOLOConfig:
    """Configuration du Composant 1 — YOLOv8."""
    model_name: str = 'yolov8m.pt'  # medium — compromis perf/mémoire
    confidence_threshold: float = 0.25
    input_size: int = 640
    num_classes: int = 80  # COCO


@dataclass
class DetectionConfig:
    """Configuration du Composant 2 — Détection multi-branches."""
    # Backbones
    person_backbone: str = 'efficientnet_b2'  # ~9M params
    context_backbone: str = 'resnet18'  # ~11M params

    # Dimensions des features
    person_feat_dim: int = 1408  # EfficientNet-B2 avgpool output
    context_feat_dim: int = 512  # ResNet-18 avgpool output
    yolo_feat_dim: int = 64  # sortie du MLP YOLO
    yolo_hidden_dim: int = 128  # couche cachée du MLP YOLO

    # Tête de fusion
    fusion_hidden_dim: int = 512
    fusion_dropout: float = 0.3

    # Input
    input_size: int = 224  # taille des images d'entrée (crops + contexte)

    # Entraînement
    batch_size: int = 32
    num_epochs: int = 25
    warmup_epochs: int = 5  # époques avec backbones gelés
    lr_head: float = 1e-4
    lr_backbone: float = 1e-5
    weight_decay: float = 1e-4
    scheduler: str = 'cosine'

    # Loss
    vad_loss_weight: float = 0.5  # lambda dans L = L_BCE + λ * L_MSE
    use_focal_loss: bool = True
    focal_alpha: float = 0.25
    focal_gamma: float = 2.0

    # Data augmentation
    use_augmentation: bool = True

    # Nombre de workers DataLoader
    num_workers: int = 2


@dataclass
class GenerationConfig:
    """Configuration du Composant 3 — Génération conditionnelle."""
    # Modèle de base
    base_model: str = 'runwayml/stable-diffusion-v1-5'

    # LoRA
    lora_rank: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.1
    lora_target_modules: List[str] = field(
        default_factory=lambda: ['to_q', 'to_v', 'to_k', 'to_out.0']
    )

    # Entraînement
    num_train_images: int = 5000  # sous-ensemble EMOTIC
    batch_size: int = 1
    gradient_accumulation_steps: int = 4
    num_epochs: int = 10
    lr: float = 1e-4
    image_size: int = 512

    # Génération
    num_inference_steps: int = 50
    guidance_scale: float = 7.5
    num_eval_images: int = 500


@dataclass
class Config:
    """Configuration globale."""
    paths: PathConfig = field(default_factory=PathConfig)
    yolo: YOLOConfig = field(default_factory=YOLOConfig)
    detection: DetectionConfig = field(default_factory=DetectionConfig)
    generation: GenerationConfig = field(default_factory=GenerationConfig)
    seed: int = 42
    device: str = 'cuda'

    def __post_init__(self):
        self.paths.makedirs()


# Instance globale
cfg = Config()


---
## Section 1 — Téléchargement du dataset EMOTIC

Le dataset EMOTIC n'est pas directement téléchargeable par script. Deux options :

**Option A — Google Drive (recommandé) :**
1. Téléchargez EMOTIC depuis http://sunai.uoc.edu/emotic
2. Uploadez le zip sur votre Google Drive
3. Exécutez la cellule ci-dessous

**Option B — Kaggle :**
Si EMOTIC est disponible sur Kaggle, adaptez la cellule.

**Structure attendue :**
```
/content/datasets/emotic/
├── Annotations/
│   └── Annotations.mat
├── emotic/
│   └── images/
└── mscoco/
    └── images/
```

In [ ]:
# === Option A : depuis Google Drive ===
# Décommentez les 3 lignes suivantes :

# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -q /content/drive/MyDrive/emotic.zip -d /content/datasets/emotic

# === Vérification ===
emotic_check = os.path.exists(cfg.paths.emotic_mat_file)
print(f"EMOTIC Annotations.mat trouvé : {emotic_check}")
if not emotic_check:
    print("⚠️  Téléchargez EMOTIC avant de continuer (voir instructions ci-dessus)")
else:
    # Afficher la structure
    for root, dirs, files in os.walk(cfg.paths.emotic_root):
        level = root.replace(cfg.paths.emotic_root, '').count(os.sep)
        if level < 2:
            indent = ' ' * 2 * level
            print(f'{indent}{os.path.basename(root)}/')
            subindent = ' ' * 2 * (level + 1)
            for f in files[:5]:
                print(f'{subindent}{f}')
            if len(files) > 5:
                print(f'{subindent}... ({len(files)} fichiers)')

---
## Section 2 — Préparation des données

Parse le fichier `Annotations.mat` d'EMOTIC et génère des CSV exploitables avec :
- Bounding boxes des personnes
- 26 catégories binaires (multi-label)
- 3 dimensions VAD continues
- Poids de classes pour la Focal Loss

In [ ]:
"""
Préparation des données EMOTIC.
Parse le fichier Annotations.mat et génère des CSV exploitables.

Usage sur Colab :
    !python prepare_data.py

Le dataset EMOTIC doit être téléchargé manuellement et extrait dans
le dossier configuré dans config.py (par défaut /content/datasets/emotic/).
Structure attendue :
    emotic/
    ├── Annotations/
    │   └── Annotations.mat
    ├── emotic/          # ou emotic_subfolders selon la version
    │   ├── images/
    │   └── ...
    └── mscoco/
        └── images/
"""

import os
import csv
import numpy as np
from scipy.io import loadmat
from config import cfg, EMOTIC_CATEGORIES


def parse_emotic_mat(mat_path: str):
    """
    Parse le fichier Annotations.mat d'EMOTIC.
    Retourne une liste de dicts par split (train, val, test).
    
    Chaque entrée contient :
        - image_folder: sous-dossier de l'image (ex: 'mscoco/images')
        - image_file: nom du fichier image
        - person_bbox: [x1, y1, x2, y2]
        - categories: liste de catégories (strings)
        - vad: [valence, arousal, dominance] ou None
    """
    print(f"[DATA] Chargement de {mat_path}...")
    mat = loadmat(mat_path, squeeze_me=True, struct_as_record=False)

    splits = {}
    for split_name in ['train', 'val', 'test']:
        key = split_name
        if key not in mat:
            # Certaines versions du .mat utilisent des clés légèrement différentes
            for k in mat.keys():
                if split_name in k.lower() and not k.startswith('__'):
                    key = k
                    break

        if key not in mat:
            print(f"  [WARN] Split '{split_name}' non trouvé dans le .mat, skip.")
            continue

        split_data = mat[key]
        records = []

        # split_data est un array de structs
        if not hasattr(split_data, '__len__'):
            split_data = [split_data]

        for img_struct in split_data:
            # Extraction des champs image
            try:
                image_folder = str(img_struct.folder) if hasattr(img_struct, 'folder') else ''
                image_file = str(img_struct.filename) if hasattr(img_struct, 'filename') else ''
            except Exception:
                continue

            # Chaque image peut contenir plusieurs personnes annotées
            persons = img_struct.person if hasattr(img_struct, 'person') else []
            if not hasattr(persons, '__len__') or isinstance(persons, str):
                persons = [persons]

            for person in persons:
                record = {
                    'image_folder': image_folder,
                    'image_file': image_file,
                    'person_bbox': [0, 0, 0, 0],
                    'categories': [],
                    'valence': -1.0,
                    'arousal': -1.0,
                    'dominance': -1.0,
                }

                # Bounding box
                try:
                    body = person.body if hasattr(person, 'body') else person
                    bbox = body.bbox if hasattr(body, 'bbox') else None
                    if bbox is not None:
                        bbox = np.array(bbox).flatten().tolist()
                        if len(bbox) >= 4:
                            record['person_bbox'] = [int(b) for b in bbox[:4]]
                except Exception:
                    pass

                # Catégories discrètes
                try:
                    cats = person.combined_categories if hasattr(person, 'combined_categories') else []
                    if hasattr(person, 'annotations_categories'):
                        cats = person.annotations_categories
                    if isinstance(cats, str):
                        cats = [cats]
                    elif hasattr(cats, 'tolist'):
                        cats = cats.tolist()
                    if not isinstance(cats, list):
                        cats = list(cats) if hasattr(cats, '__iter__') else []
                    # Filtrer pour ne garder que les catégories EMOTIC valides
                    record['categories'] = [c for c in cats if c in EMOTIC_CATEGORIES]
                except Exception:
                    pass

                # VAD continu
                try:
                    cav = person.combined_continuous if hasattr(person, 'combined_continuous') else None
                    if cav is None and hasattr(person, 'annotations_continuous'):
                        cav = person.annotations_continuous
                    if cav is not None:
                        if hasattr(cav, 'valence'):
                            record['valence'] = float(cav.valence)
                            record['arousal'] = float(cav.arousal)
                            record['dominance'] = float(cav.dominance)
                        elif hasattr(cav, '__len__') and len(cav) >= 3:
                            vals = np.array(cav).flatten()
                            record['valence'] = float(vals[0])
                            record['arousal'] = float(vals[1])
                            record['dominance'] = float(vals[2])
                except Exception:
                    pass

                records.append(record)

        splits[split_name] = records
        print(f"  [DATA] {split_name}: {len(records)} annotations extraites")

    return splits


def save_splits_csv(splits: dict, output_dir: str):
    """Sauvegarde chaque split en CSV."""
    os.makedirs(output_dir, exist_ok=True)

    for split_name, records in splits.items():
        csv_path = os.path.join(output_dir, f'{split_name}.csv')
        with open(csv_path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            # Header
            header = ['image_folder', 'image_file',
                      'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2',
                      'valence', 'arousal', 'dominance']
            header += EMOTIC_CATEGORIES  # 26 colonnes binaires
            writer.writerow(header)

            for rec in records:
                row = [
                    rec['image_folder'],
                    rec['image_file'],
                    *rec['person_bbox'],
                    rec['valence'],
                    rec['arousal'],
                    rec['dominance'],
                ]
                # One-hot des catégories
                for cat in EMOTIC_CATEGORIES:
                    row.append(1 if cat in rec['categories'] else 0)
                writer.writerow(row)

        print(f"[DATA] Sauvegardé : {csv_path} ({len(records)} lignes)")


def compute_class_weights(csv_path: str) -> dict:
    """
    Calcule les poids de classes pour la Focal/BCE Loss
    basés sur la fréquence inverse dans le split train.
    """
    import pandas as pd
    df = pd.read_csv(csv_path)
    cat_cols = EMOTIC_CATEGORIES
    counts = df[cat_cols].sum()
    total = len(df)

    weights = {}
    for cat in cat_cols:
        freq = counts[cat] / total if total > 0 else 0.5
        # Poids inverse lissé
        weights[cat] = 1.0 / (freq + 1e-6)

    # Normaliser pour que le poids moyen soit 1
    mean_w = np.mean(list(weights.values()))
    weights = {k: v / mean_w for k, v in weights.items()}

    print("\n[DATA] Poids de classes (top 5 les plus rares) :")
    sorted_w = sorted(weights.items(), key=lambda x: x[1], reverse=True)
    for cat, w in sorted_w[:5]:
        print(f"  {cat}: {w:.2f} (fréq: {counts[cat]}/{total})")

    return weights




In [ ]:
# === Exécuter la préparation ===
mat_path = cfg.paths.emotic_mat_file
output_dir = os.path.join(cfg.paths.project_root, 'csv_annotations')

splits = parse_emotic_mat(mat_path)
save_splits_csv(splits, output_dir)

# Poids de classes
train_csv = os.path.join(output_dir, 'train.csv')
weights = compute_class_weights(train_csv)
with open(os.path.join(output_dir, 'class_weights.json'), 'w') as f:
    json.dump(weights, f, indent=2)

print("\n✅ Préparation des données terminée.")

### Visualisation de la distribution des données

In [ ]:
# Distribution des 26 catégories EMOTIC
df_train = pd.read_csv(os.path.join(cfg.paths.project_root, 'csv_annotations', 'train.csv'))
counts = df_train[EMOTIC_CATEGORIES].sum().sort_values(ascending=True)

plt.figure(figsize=(10, 8))
counts.plot(kind='barh', color='steelblue')
plt.xlabel("Nombre d'annotations")
plt.title('Distribution des 26 catégories EMOTIC (train)')
plt.tight_layout()
plt.savefig(os.path.join(cfg.paths.log_dir, 'class_distribution.png'), dpi=150)
plt.show()

# Distribution VAD
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for i, (col, title) in enumerate([('valence', 'Valence'),
                                   ('arousal', 'Arousal'),
                                   ('dominance', 'Dominance')]):
    valid = df_train[df_train[col] >= 0][col]
    axes[i].hist(valid, bins=50, color='coral', alpha=0.8)
    axes[i].set_title(f'{title} (n={len(valid)})')
    axes[i].set_xlabel('Score')
plt.suptitle('Distribution VAD (train)', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(cfg.paths.log_dir, 'vad_distribution.png'), dpi=150)
plt.show()

print(f"\nTrain : {len(df_train)} samples")
print(f"Catégorie la + fréquente : {counts.index[-1]} ({int(counts.iloc[-1])})")
print(f"Catégorie la - fréquente : {counts.index[0]} ({int(counts.iloc[0])})")

---
## Section 3 — Composant 1 : Extraction des features YOLO

**YOLOv8-medium** pré-entraîné sur COCO (80 classes) est utilisé en inférence pure pour extraire un **vecteur contextuel de dimension 80** par image (profil d'objets de la scène).

- Pas d'entraînement YOLO — uniquement inférence
- Vecteurs pré-calculés et stockés en `.pt` pour dissocier le coût YOLO du budget d'entraînement
- Temps estimé : **~10 minutes** sur T4 pour ~23k images

In [ ]:
"""
Composant 1 — Extraction de features contextuelles avec YOLOv8.

Effectue l'inférence YOLOv8m sur toutes les images EMOTIC et 
sauvegarde un vecteur 80D (profil d'objets COCO) par image.

Usage :
    python extract_yolo_features.py

Temps estimé : ~8-10 min sur T4 pour ~23k images.
"""

import os
import glob
import time
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

from config import cfg, NUM_COCO_CLASSES


def extract_yolo_features_for_dataset():
    """
    Pour chaque image unique dans EMOTIC, exécute YOLOv8 et 
    sauvegarde un vecteur de dimension 80 (classes COCO).
    """
    from ultralytics import YOLO

    # ─── Chargement du modèle ───
    print(f"[YOLO] Chargement de {cfg.yolo.model_name}...")
    model = YOLO(cfg.yolo.model_name)
    print(f"[YOLO] Modèle chargé ({cfg.yolo.model_name})")

    # ─── Collecte de toutes les images uniques depuis les CSV ───
    csv_dir = os.path.join(cfg.paths.project_root, 'csv_annotations')
    all_images = set()

    for split in ['train', 'val', 'test']:
        csv_path = os.path.join(csv_dir, f'{split}.csv')
        if not os.path.exists(csv_path):
            print(f"  [WARN] {csv_path} non trouvé, skip.")
            continue
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            all_images.add((str(row['image_folder']), str(row['image_file'])))

    print(f"[YOLO] {len(all_images)} images uniques à traiter")

    # ─── Dossier de sortie ───
    output_dir = cfg.paths.yolo_features_dir
    os.makedirs(output_dir, exist_ok=True)

    # ─── Statistiques ───
    stats = {
        'total_images': len(all_images),
        'processed': 0,
        'failed': 0,
        'objects_per_image': [],
        'class_counts': defaultdict(int),
    }

    start_time = time.time()

    for i, (folder, filename) in enumerate(all_images):
        # Clé de fichier pour la sauvegarde
        feat_key = filename.replace('/', '_').replace('.', '_')
        feat_path = os.path.join(output_dir, f'{feat_key}.pt')

        # Skip si déjà calculé
        if os.path.exists(feat_path):
            stats['processed'] += 1
            continue

        # Chemin complet de l'image
        img_path = os.path.join(cfg.paths.emotic_images_root, folder, filename)
        if not os.path.exists(img_path):
            # Tenter d'autres sous-dossiers
            found = False
            for sub in ['mscoco/images', 'emotic/images', 'framesdb', 'ade20k']:
                alt = os.path.join(cfg.paths.emotic_images_root, sub, filename)
                if os.path.exists(alt):
                    img_path = alt
                    found = True
                    break
            if not found:
                stats['failed'] += 1
                # Sauvegarder un vecteur nul
                torch.save(torch.zeros(NUM_COCO_CLASSES, dtype=torch.float32), feat_path)
                continue

        # ─── Inférence YOLO ───
        try:
            results = model(img_path, verbose=False,
                            conf=cfg.yolo.confidence_threshold,
                            imgsz=cfg.yolo.input_size)

            # Construction du vecteur 80D
            feature_vector = torch.zeros(NUM_COCO_CLASSES, dtype=torch.float32)

            if len(results) > 0 and results[0].boxes is not None:
                boxes = results[0].boxes
                classes = boxes.cls.cpu().numpy().astype(int)
                confs = boxes.conf.cpu().numpy()

                num_objects = len(classes)
                stats['objects_per_image'].append(num_objects)

                for cls_id, conf in zip(classes, confs):
                    if 0 <= cls_id < NUM_COCO_CLASSES:
                        # Max pooling des confiances par classe
                        feature_vector[cls_id] = max(feature_vector[cls_id].item(), conf)
                        stats['class_counts'][int(cls_id)] += 1
            else:
                stats['objects_per_image'].append(0)

            torch.save(feature_vector, feat_path)
            stats['processed'] += 1

        except Exception as e:
            print(f"  [ERR] {filename}: {e}")
            torch.save(torch.zeros(NUM_COCO_CLASSES, dtype=torch.float32), feat_path)
            stats['failed'] += 1

        # Progress
        if (i + 1) % 1000 == 0:
            elapsed = time.time() - start_time
            rate = (i + 1) / elapsed
            remaining = (len(all_images) - i - 1) / rate
            print(f"  [{i+1}/{len(all_images)}] "
                  f"{rate:.1f} img/s — restant: {remaining/60:.1f} min")

    elapsed = time.time() - start_time

    # ─── Rapport ───
    print(f"\n{'='*60}")
    print(f"[YOLO] EXTRACTION TERMINÉE")
    print(f"{'='*60}")
    print(f"  Images traitées : {stats['processed']}")
    print(f"  Échecs          : {stats['failed']}")
    print(f"  Temps total     : {elapsed:.1f}s ({elapsed/60:.1f} min)")
    if stats['objects_per_image']:
        arr = np.array(stats['objects_per_image'])
        print(f"  Objets/image    : moy={arr.mean():.1f}, "
              f"méd={np.median(arr):.0f}, max={arr.max()}")
        print(f"  Images sans obj : {(arr == 0).sum()} "
              f"({(arr == 0).mean()*100:.1f}%)")

    # Top 10 classes détectées
    if stats['class_counts']:
        # Charger les noms COCO
        coco_names = model.names if hasattr(model, 'names') else {}
        print(f"\n  Top 10 classes COCO détectées :")
        sorted_classes = sorted(stats['class_counts'].items(),
                                key=lambda x: x[1], reverse=True)
        for cls_id, count in sorted_classes[:10]:
            name = coco_names.get(cls_id, f'class_{cls_id}')
            print(f"    {name:20s} : {count:6d} détections")

    # Sauvegarder les stats
    stats_summary = {
        'total_images': stats['total_images'],
        'processed': stats['processed'],
        'failed': stats['failed'],
        'mean_objects': float(np.mean(stats['objects_per_image'])) if stats['objects_per_image'] else 0,
        'median_objects': float(np.median(stats['objects_per_image'])) if stats['objects_per_image'] else 0,
    }
    import json
    stats_path = os.path.join(output_dir, 'extraction_stats.json')
    with open(stats_path, 'w') as f:
        json.dump(stats_summary, f, indent=2)
    print(f"\n  Stats sauvegardées dans {stats_path}")




In [ ]:
# === Lancer l'extraction YOLO ===
extract_yolo_features_for_dataset()

---
## Dataset PyTorch & DataLoader

Dataset EMOTIC multi-branches avec :
- Crop de la personne (bbox EMOTIC) → branche 1
- Image contexte complète → branche 2
- Vecteur YOLO 80D pré-calculé → branche 3
- Labels multi-label (26 catégories) + VAD (3 continu)
- Augmentation via Albumentations
- WeightedRandomSampler pour le déséquilibre de classes

In [ ]:
"""
Dataset PyTorch pour EMOTIC.
Gère le chargement des crops personne, images contexte, features YOLO,
et les labels multi-label + VAD.
"""

import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

from config import cfg, EMOTIC_CATEGORIES, NUM_CATEGORIES, NUM_VAD


def get_transforms(split: str, input_size: int = 224):
    """
    Retourne les transformations Albumentations pour un split donné.
    On utilise Albumentations plutôt que torchvision.transforms pour
    sa flexibilité et ses augmentations spatiales cohérentes.
    """
    if split == 'train' and cfg.detection.use_augmentation:
        return A.Compose([
            A.Resize(input_size, input_size),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=15, p=0.3),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.4),
            A.RandomResizedCrop(height=input_size, width=input_size,
                                scale=(0.85, 1.0), ratio=(0.9, 1.1), p=0.3),
            A.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(input_size, input_size),
            A.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])


class EMOTICDataset(Dataset):
    """
    Dataset EMOTIC pour l'entraînement multi-branches.

    Retourne pour chaque sample :
        - person_crop: tensor [3, 224, 224] — crop de la personne
        - context_img: tensor [3, 224, 224] — image complète
        - yolo_features: tensor [80] — vecteur YOLO pré-calculé
        - cat_labels: tensor [26] — one-hot multi-label
        - vad_labels: tensor [3] — [valence, arousal, dominance] normalisés [0,1]
        - vad_mask: tensor [3] — 1.0 si VAD annoté, 0.0 sinon
    """

    def __init__(self, csv_path: str, split: str = 'train',
                 yolo_features_dir: str = None):
        """
        Args:
            csv_path: chemin vers le CSV du split (généré par prepare_data.py)
            split: 'train', 'val', ou 'test'
            yolo_features_dir: dossier contenant les vecteurs YOLO pré-calculés (.pt)
        """
        self.df = pd.read_csv(csv_path)
        self.split = split
        self.yolo_features_dir = yolo_features_dir or cfg.paths.yolo_features_dir
        self.images_root = cfg.paths.emotic_images_root
        self.input_size = cfg.detection.input_size

        self.transform = get_transforms(split, self.input_size)

        # Vérification
        print(f"[Dataset] {split}: {len(self.df)} samples chargés")
        cat_counts = self.df[EMOTIC_CATEGORIES].sum()
        print(f"  Catégories les + fréquentes : {cat_counts.nlargest(3).to_dict()}")
        print(f"  Catégories les - fréquentes : {cat_counts.nsmallest(3).to_dict()}")

    def __len__(self):
        return len(self.df)

    def _load_image(self, folder: str, filename: str) -> np.ndarray:
        """Charge une image et retourne un array RGB."""
        img_path = os.path.join(self.images_root, folder, filename)
        if not os.path.exists(img_path):
            # Fallback : chercher dans les sous-dossiers courants
            for subfolder in ['mscoco/images', 'emotic/images', 'framesdb', 'ade20k']:
                alt_path = os.path.join(self.images_root, subfolder, filename)
                if os.path.exists(alt_path):
                    img_path = alt_path
                    break

        try:
            img = Image.open(img_path).convert('RGB')
            return np.array(img)
        except Exception as e:
            # Image placeholder en cas d'erreur
            print(f"  [WARN] Image non trouvée : {img_path}")
            return np.zeros((self.input_size, self.input_size, 3), dtype=np.uint8)

    def _crop_person(self, img: np.ndarray, bbox: list) -> np.ndarray:
        """Extrait le crop de la personne à partir du bounding box."""
        x1, y1, x2, y2 = bbox
        h, w = img.shape[:2]

        # Clamp aux dimensions de l'image
        x1 = max(0, min(x1, w - 1))
        x2 = max(x1 + 1, min(x2, w))
        y1 = max(0, min(y1, h - 1))
        y2 = max(y1 + 1, min(y2, h))

        crop = img[y1:y2, x1:x2]

        # Vérifier que le crop n'est pas dégénéré
        if crop.shape[0] < 2 or crop.shape[1] < 2:
            return img  # fallback : utiliser l'image entière
        return crop

    def _load_yolo_features(self, folder: str, filename: str) -> torch.Tensor:
        """Charge le vecteur YOLO pré-calculé pour cette image."""
        # Clé unique basée sur le nom de fichier
        feat_key = filename.replace('/', '_').replace('.', '_')
        feat_path = os.path.join(self.yolo_features_dir, f'{feat_key}.pt')

        if os.path.exists(feat_path):
            return torch.load(feat_path, map_location='cpu', weights_only=True)
        else:
            # Vecteur nul si pas encore pré-calculé
            return torch.zeros(cfg.yolo.num_classes, dtype=torch.float32)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # ─── Chargement image complète ───
        folder = str(row['image_folder'])
        filename = str(row['image_file'])
        img = self._load_image(folder, filename)

        # ─── Crop personne ───
        bbox = [int(row['bbox_x1']), int(row['bbox_y1']),
                int(row['bbox_x2']), int(row['bbox_y2'])]
        person_crop = self._crop_person(img, bbox)

        # ─── Transformations ───
        person_transformed = self.transform(image=person_crop)['image']
        context_transformed = self.transform(image=img)['image']

        # ─── Features YOLO ───
        yolo_features = self._load_yolo_features(folder, filename)

        # ─── Labels catégoriels (multi-label) ───
        cat_labels = torch.tensor(
            [float(row[cat]) for cat in EMOTIC_CATEGORIES],
            dtype=torch.float32
        )

        # ─── Labels VAD ───
        valence = float(row['valence'])
        arousal = float(row['arousal'])
        dominance = float(row['dominance'])

        # Normalisation [0, 10] → [0, 1]
        vad_labels = torch.tensor([valence / 10.0, arousal / 10.0, dominance / 10.0],
                                  dtype=torch.float32)

        # Masque pour les VAD manquants (valeur -1 dans le CSV)
        vad_mask = torch.tensor([
            1.0 if valence >= 0 else 0.0,
            1.0 if arousal >= 0 else 0.0,
            1.0 if dominance >= 0 else 0.0,
        ], dtype=torch.float32)

        # Clamp les valeurs VAD valides
        vad_labels = vad_labels.clamp(0.0, 1.0)

        return {
            'person_crop': person_transformed,
            'context_img': context_transformed,
            'yolo_features': yolo_features,
            'cat_labels': cat_labels,
            'vad_labels': vad_labels,
            'vad_mask': vad_mask,
            'image_file': filename,
        }


def build_dataloader(csv_path: str, split: str, yolo_features_dir: str = None,
                     batch_size: int = None, num_workers: int = None) -> DataLoader:
    """
    Construit un DataLoader pour un split donné.
    Pour le train, applique un WeightedRandomSampler pour le déséquilibre.
    """
    bs = batch_size or cfg.detection.batch_size
    nw = num_workers if num_workers is not None else cfg.detection.num_workers

    dataset = EMOTICDataset(csv_path, split=split, yolo_features_dir=yolo_features_dir)

    sampler = None
    shuffle = (split == 'train')

    if split == 'train':
        # Calcul des poids d'échantillonnage basés sur la rareté des catégories
        cat_cols = EMOTIC_CATEGORIES
        cat_matrix = dataset.df[cat_cols].values  # [N, 26]
        # Fréquences par catégorie
        freqs = cat_matrix.sum(axis=0) / len(cat_matrix)
        freqs = np.clip(freqs, 1e-6, None)
        # Poids par sample = max des poids inverses de ses catégories actives
        inv_freqs = 1.0 / freqs
        sample_weights = (cat_matrix * inv_freqs).max(axis=1)
        sample_weights = sample_weights / sample_weights.mean()

        sampler = WeightedRandomSampler(
            weights=sample_weights.tolist(),
            num_samples=len(dataset),
            replacement=True
        )
        shuffle = False  # sampler et shuffle sont mutuellement exclusifs

    loader = DataLoader(
        dataset,
        batch_size=bs,
        shuffle=shuffle,
        sampler=sampler,
        num_workers=nw,
        pin_memory=True,
        drop_last=(split == 'train'),
    )

    return loader




In [ ]:
# === Test rapide du Dataset ===
train_csv_path = os.path.join(cfg.paths.project_root, 'csv_annotations', 'train.csv')
loader = build_dataloader(train_csv_path, 'train', batch_size=4)
batch = next(iter(loader))
print(f"\n✅ Batch shapes :")
print(f"  person_crop  : {batch['person_crop'].shape}")
print(f"  context_img  : {batch['context_img'].shape}")
print(f"  yolo_features: {batch['yolo_features'].shape}")
print(f"  cat_labels   : {batch['cat_labels'].shape}")
print(f"  vad_labels   : {batch['vad_labels'].shape}")

---
## Section 4 — Composant 2 : Modèle multi-branches & Entraînement

### Architecture

| Branche | Backbone | Params | Sortie |
|---------|----------|--------|--------|
| 1 — Personne | EfficientNet-B2 (ImageNet) | ~9M | 1408D |
| 2 — Contexte | ResNet-18 (ImageNet) | ~11M | 512D |
| 3 — YOLO | MLP 80→128→64 | ~11K | 64D |
| **Fusion** | Concat → FC 1984→512→256 | ~1.3M | 26 cats + 3 VAD |

**Loss** : `L = Focal_Loss(26 cats) + λ · MSE(VAD)` avec λ = 0.5

**Entraînement** en 2 phases :
1. **Warm-up** (5 époques) : backbones gelés, seule la tête est entraînée
2. **Fine-tuning** (20 époques) : tout dégelé avec lr différencié (backbones × 0.1)

In [ ]:
"""
Composant 2 — Architecture multi-branches pour la détection d'émotions.

Trois branches :
  1. EfficientNet-B2 sur le crop personne → 1408D
  2. ResNet-18 sur l'image contexte → 512D
  3. MLP sur le vecteur YOLO 80D → 64D

Fusion par concaténation → tête FC → 26 catégories + 3 VAD.
"""

import torch
import torch.nn as nn
import torchvision.models as models

from config import (
    cfg, NUM_CATEGORIES, NUM_VAD, NUM_COCO_CLASSES,
)


class YOLOFeatureMLP(nn.Module):
    """
    Branche 3 : projette le vecteur YOLO 80D dans un espace
    de dimension compatible avec les branches visuelles.
    """
    def __init__(self, input_dim=NUM_COCO_CLASSES,
                 hidden_dim=128, output_dim=64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, output_dim),
            nn.BatchNorm1d(output_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.mlp(x)


class FusionHead(nn.Module):
    """
    Tête de fusion : concatène les features des 3 branches
    et prédit les 26 catégories + 3 VAD.
    """
    def __init__(self, input_dim, hidden_dim=512, dropout=0.3,
                 num_categories=NUM_CATEGORIES, num_vad=NUM_VAD):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
        )
        # Tête classification multi-label
        self.cat_head = nn.Linear(hidden_dim // 2, num_categories)
        # Tête régression VAD
        self.vad_head = nn.Sequential(
            nn.Linear(hidden_dim // 2, num_vad),
            nn.Sigmoid(),  # sortie dans [0, 1]
        )

    def forward(self, x):
        shared = self.shared(x)
        cat_logits = self.cat_head(shared)  # [B, 26] — logits bruts
        vad_pred = self.vad_head(shared)    # [B, 3] — [0, 1]
        return cat_logits, vad_pred


class EmotionDetector(nn.Module):
    """
    Modèle complet de détection d'émotions contextuelles.

    Architecture :
        Branche 1 (personne) : EfficientNet-B2 → 1408D
        Branche 2 (contexte) : ResNet-18 → 512D
        Branche 3 (YOLO)     : MLP 80→64D
        Fusion : concat(1408+512+64=1984) → FC → 26 cats + 3 VAD
    """

    def __init__(self, config=None):
        super().__init__()
        c = config or cfg.detection

        # ─── Branche 1 : Personne (EfficientNet-B2) ───
        if c.person_backbone == 'efficientnet_b2':
            efficientnet = models.efficientnet_b2(weights='IMAGENET1K_V1')
            self.person_backbone = nn.Sequential(*list(efficientnet.children())[:-1])
            # Flatten après adaptive avg pool
            self.person_flatten = nn.Flatten()
            person_out_dim = c.person_feat_dim  # 1408
        else:
            raise ValueError(f"Backbone personne non supporté : {c.person_backbone}")

        # ─── Branche 2 : Contexte (ResNet-18) ───
        if c.context_backbone == 'resnet18':
            resnet = models.resnet18(weights='IMAGENET1K_V1')
            # Retirer la couche FC finale
            self.context_backbone = nn.Sequential(*list(resnet.children())[:-1])
            self.context_flatten = nn.Flatten()
            context_out_dim = c.context_feat_dim  # 512
        else:
            raise ValueError(f"Backbone contexte non supporté : {c.context_backbone}")

        # ─── Branche 3 : YOLO MLP ───
        self.yolo_branch = YOLOFeatureMLP(
            input_dim=NUM_COCO_CLASSES,
            hidden_dim=c.yolo_hidden_dim,
            output_dim=c.yolo_feat_dim,
        )
        yolo_out_dim = c.yolo_feat_dim  # 64

        # ─── Fusion ───
        fusion_input_dim = person_out_dim + context_out_dim + yolo_out_dim
        self.fusion_head = FusionHead(
            input_dim=fusion_input_dim,
            hidden_dim=c.fusion_hidden_dim,
            dropout=c.fusion_dropout,
        )

        # Dimensions pour le log
        self._dims = {
            'person': person_out_dim,
            'context': context_out_dim,
            'yolo': yolo_out_dim,
            'fusion_input': fusion_input_dim,
        }

    def freeze_backbones(self):
        """Gèle les backbones pour le warm-up."""
        for param in self.person_backbone.parameters():
            param.requires_grad = False
        for param in self.context_backbone.parameters():
            param.requires_grad = False
        print("[Model] Backbones gelés (warm-up)")

    def unfreeze_backbones(self):
        """Dégèle les backbones pour le fine-tuning."""
        for param in self.person_backbone.parameters():
            param.requires_grad = True
        for param in self.context_backbone.parameters():
            param.requires_grad = True
        print("[Model] Backbones dégelés (fine-tuning)")

    def get_param_groups(self, lr_head, lr_backbone):
        """
        Retourne les groupes de paramètres avec lr différenciés
        pour l'optimiseur.
        """
        backbone_params = list(self.person_backbone.parameters()) + \
                          list(self.context_backbone.parameters())
        head_params = list(self.yolo_branch.parameters()) + \
                      list(self.fusion_head.parameters())

        return [
            {'params': backbone_params, 'lr': lr_backbone},
            {'params': head_params, 'lr': lr_head},
        ]

    def forward(self, person_crop, context_img, yolo_features):
        """
        Args:
            person_crop:  [B, 3, 224, 224]
            context_img:  [B, 3, 224, 224]
            yolo_features: [B, 80]

        Returns:
            cat_logits: [B, 26] — logits pour BCE
            vad_pred:   [B, 3]  — prédictions VAD dans [0, 1]
        """
        # Branche 1
        person_feat = self.person_backbone(person_crop)
        person_feat = self.person_flatten(person_feat)  # [B, 1408]

        # Branche 2
        context_feat = self.context_backbone(context_img)
        context_feat = self.context_flatten(context_feat)  # [B, 512]

        # Branche 3
        yolo_feat = self.yolo_branch(yolo_features)  # [B, 64]

        # Fusion
        fused = torch.cat([person_feat, context_feat, yolo_feat], dim=1)
        cat_logits, vad_pred = self.fusion_head(fused)

        return cat_logits, vad_pred

    def count_parameters(self):
        """Affiche le nombre de paramètres par composant."""
        total = 0
        for name, module in [
            ('Person backbone', self.person_backbone),
            ('Context backbone', self.context_backbone),
            ('YOLO MLP', self.yolo_branch),
            ('Fusion head', self.fusion_head),
        ]:
            n = sum(p.numel() for p in module.parameters())
            trainable = sum(p.numel() for p in module.parameters() if p.requires_grad)
            print(f"  {name:20s}: {n:>10,} params ({trainable:>10,} trainable)")
            total += n
        print(f"  {'TOTAL':20s}: {total:>10,} params")
        return total


# ─── Losses ───

class FocalLoss(nn.Module):
    """
    Focal Loss pour la classification multi-label.
    Réduit la contribution des exemples bien classifiés
    pour se concentrer sur les classes rares/difficiles.
    """
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )
        probs = torch.sigmoid(logits)
        pt = targets * probs + (1 - targets) * (1 - probs)
        focal_weight = self.alpha * (1 - pt) ** self.gamma
        loss = focal_weight * bce

        if self.reduction == 'mean':
            return loss.mean()
        elif self.reduction == 'sum':
            return loss.sum()
        return loss


class EmotionLoss(nn.Module):
    """
    Loss combinée pour la détection d'émotions :
        L = L_cat + λ * L_vad

    L_cat : Focal Loss (ou BCE) sur les 26 catégories
    L_vad : MSE sur les 3 dimensions VAD, masqué sur les valeurs manquantes
    """
    def __init__(self, vad_weight=0.5, use_focal=True,
                 focal_alpha=0.25, focal_gamma=2.0,
                 class_weights=None):
        super().__init__()
        self.vad_weight = vad_weight

        if use_focal:
            self.cat_loss_fn = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)
        else:
            self.cat_loss_fn = nn.BCEWithLogitsLoss(
                pos_weight=class_weights  # tensor [26] optionnel
            )

        self.vad_loss_fn = nn.MSELoss(reduction='none')

    def forward(self, cat_logits, vad_pred, cat_targets, vad_targets, vad_mask):
        """
        Args:
            cat_logits:   [B, 26]
            vad_pred:     [B, 3]
            cat_targets:  [B, 26]
            vad_targets:  [B, 3]
            vad_mask:     [B, 3] — 1 si annoté, 0 sinon
        """
        # Loss catégorielle
        loss_cat = self.cat_loss_fn(cat_logits, cat_targets)

        # Loss VAD (masquée)
        loss_vad_raw = self.vad_loss_fn(vad_pred, vad_targets)  # [B, 3]
        loss_vad_masked = (loss_vad_raw * vad_mask).sum() / (vad_mask.sum() + 1e-8)

        # Loss totale
        loss_total = loss_cat + self.vad_weight * loss_vad_masked

        return {
            'total': loss_total,
            'cat': loss_cat,
            'vad': loss_vad_masked,
        }




In [ ]:
# === Test du modèle ===
model_test = EmotionDetector()
model_test.count_parameters()

# Forward pass fictif
B = 4
cat_logits, vad_pred = model_test(
    torch.randn(B, 3, 224, 224),
    torch.randn(B, 3, 224, 224),
    torch.randn(B, 80)
)
print(f"\n✅ cat_logits: {cat_logits.shape}, vad_pred: {vad_pred.shape}")

# Test loss
loss_fn_test = EmotionLoss(use_focal=True)
losses = loss_fn_test(
    cat_logits, vad_pred,
    torch.randint(0, 2, (B, 26)).float(),
    torch.rand(B, 3),
    torch.ones(B, 3)
)
print(f"✅ Loss total: {losses['total']:.4f}, cat: {losses['cat']:.4f}, vad: {losses['vad']:.4f}")
del model_test

### Fonction d'évaluation

In [ ]:
"""
Évaluation du Composant 2 — Métriques EMOTIC.

Métriques :
  - Classification multi-label : AP par catégorie, mAP
  - Régression VAD : MAE et corrélation de Pearson par dimension

Compatible avec torchmetrics pour les AP.
"""

import torch
import numpy as np
from sklearn.metrics import average_precision_score
from scipy.stats import pearsonr



@torch.no_grad()
def evaluate_model(model, dataloader, loss_fn, device):
    """
    Évalue le modèle sur un DataLoader (val ou test).

    Returns:
        dict avec loss_total, mAP, AP par catégorie, MAE/Pearson par VAD.
    """
    model.eval()

    all_cat_logits = []
    all_cat_labels = []
    all_vad_preds = []
    all_vad_labels = []
    all_vad_masks = []
    running_loss = 0.0
    num_batches = 0

    for batch in dataloader:
        person = batch['person_crop'].to(device)
        context = batch['context_img'].to(device)
        yolo = batch['yolo_features'].to(device)
        cat_labels = batch['cat_labels'].to(device)
        vad_labels = batch['vad_labels'].to(device)
        vad_mask = batch['vad_mask'].to(device)

        cat_logits, vad_pred = model(person, context, yolo)

        losses = loss_fn(cat_logits, vad_pred, cat_labels, vad_labels, vad_mask)
        running_loss += losses['total'].item()
        num_batches += 1

        all_cat_logits.append(cat_logits.cpu())
        all_cat_labels.append(cat_labels.cpu())
        all_vad_preds.append(vad_pred.cpu())
        all_vad_labels.append(vad_labels.cpu())
        all_vad_masks.append(vad_mask.cpu())

    # ─── Concaténation ───
    cat_logits = torch.cat(all_cat_logits, dim=0).numpy()    # [N, 26]
    cat_labels = torch.cat(all_cat_labels, dim=0).numpy()    # [N, 26]
    vad_preds = torch.cat(all_vad_preds, dim=0).numpy()      # [N, 3]
    vad_labels = torch.cat(all_vad_labels, dim=0).numpy()    # [N, 3]
    vad_masks = torch.cat(all_vad_masks, dim=0).numpy()      # [N, 3]

    cat_probs = 1.0 / (1.0 + np.exp(-cat_logits))  # sigmoid

    # ─── AP par catégorie ───
    ap_per_class = {}
    for i, cat_name in enumerate(EMOTIC_CATEGORIES):
        if cat_labels[:, i].sum() > 0:
            ap = average_precision_score(cat_labels[:, i], cat_probs[:, i])
            ap_per_class[cat_name] = ap
        else:
            ap_per_class[cat_name] = 0.0

    mAP = np.mean(list(ap_per_class.values()))

    # ─── MAE et Pearson pour VAD ───
    vad_names = ['valence', 'arousal', 'dominance']
    mae_per_dim = {}
    pearson_per_dim = {}

    for i, name in enumerate(vad_names):
        mask = vad_masks[:, i] > 0.5
        if mask.sum() > 10:
            pred_masked = vad_preds[mask, i]
            label_masked = vad_labels[mask, i]
            mae_per_dim[name] = float(np.mean(np.abs(pred_masked - label_masked)))
            r, _ = pearsonr(pred_masked, label_masked)
            pearson_per_dim[name] = float(r) if not np.isnan(r) else 0.0
        else:
            mae_per_dim[name] = -1.0
            pearson_per_dim[name] = 0.0

    # ─── Résultats ───
    results = {
        'loss_total': running_loss / max(num_batches, 1),
        'mAP': mAP,
        'ap_per_class': ap_per_class,
        'mae_v': mae_per_dim.get('valence', -1),
        'mae_a': mae_per_dim.get('arousal', -1),
        'mae_d': mae_per_dim.get('dominance', -1),
        'pearson_v': pearson_per_dim.get('valence', 0),
        'pearson_a': pearson_per_dim.get('arousal', 0),
        'pearson_d': pearson_per_dim.get('dominance', 0),
    }

    return results


def print_full_evaluation(results: dict):
    """Affiche les résultats d'évaluation de façon détaillée."""
    print(f"\n{'='*60}")
    print(f"RÉSULTATS D'ÉVALUATION")
    print(f"{'='*60}")

    print(f"\n  Loss totale : {results['loss_total']:.4f}")
    print(f"  mAP (26 cat): {results['mAP']:.4f}")

    print(f"\n  AP par catégorie :")
    ap = results['ap_per_class']
    # Trier par AP décroissant
    for cat, val in sorted(ap.items(), key=lambda x: x[1], reverse=True):
        bar = '█' * int(val * 40)
        print(f"    {cat:22s} : {val:.4f} {bar}")

    print(f"\n  Régression VAD :")
    for dim, short in [('valence', 'v'), ('arousal', 'a'), ('dominance', 'd')]:
        mae = results[f'mae_{short}']
        r = results[f'pearson_{short}']
        if mae >= 0:
            print(f"    {dim:10s} : MAE={mae:.4f}, Pearson r={r:.4f}")
        else:
            print(f"    {dim:10s} : pas assez de données annotées")




### Boucle d'entraînement

In [ ]:
"""
Entraînement du Composant 2 — Détecteur d'émotions multi-branches.

Usage :
    python train.py

Workflow :
    1. Warm-up (5 époques) : backbones gelés, seuls la tête de fusion
       et le MLP YOLO sont entraînés
    2. Fine-tuning (20 époques) : tout est dégelé avec lr réduit
       pour les backbones

Temps estimé : ~2-3h sur T4 (Colab Pro)
"""

import time
import json
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import numpy as np

# Ajouter le dossier racine au path



def train_one_epoch(model, dataloader, loss_fn, optimizer, device, epoch):
    """Entraîne le modèle pour une époque."""
    model.train()
    running_losses = {'total': 0, 'cat': 0, 'vad': 0}
    num_batches = 0

    for batch_idx, batch in enumerate(dataloader):
        person = batch['person_crop'].to(device)
        context = batch['context_img'].to(device)
        yolo = batch['yolo_features'].to(device)
        cat_labels = batch['cat_labels'].to(device)
        vad_labels = batch['vad_labels'].to(device)
        vad_mask = batch['vad_mask'].to(device)

        # Forward
        cat_logits, vad_pred = model(person, context, yolo)

        # Loss
        losses = loss_fn(cat_logits, vad_pred, cat_labels, vad_labels, vad_mask)

        # Backward
        optimizer.zero_grad()
        losses['total'].backward()

        # Gradient clipping
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        # Accumulation
        for k in running_losses:
            running_losses[k] += losses[k].item()
        num_batches += 1

        # Log
        if (batch_idx + 1) % 50 == 0:
            avg = running_losses['total'] / num_batches
            print(f"  Epoch {epoch} [{batch_idx+1}/{len(dataloader)}] "
                  f"loss={avg:.4f}")

    # Moyennes
    for k in running_losses:
        running_losses[k] /= max(num_batches, 1)

    return running_losses


def train():
    """Boucle d'entraînement complète."""
    device = torch.device(cfg.device if torch.cuda.is_available() else 'cpu')
    print(f"[TRAIN] Device : {device}")

    # ─── Seed ───
    torch.manual_seed(cfg.seed)
    np.random.seed(cfg.seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(cfg.seed)

    # ─── Data ───
    csv_dir = os.path.join(cfg.paths.project_root, 'csv_annotations')
    train_csv = os.path.join(csv_dir, 'train.csv')
    val_csv = os.path.join(csv_dir, 'val.csv')

    if not os.path.exists(train_csv):
        print("[ERREUR] CSV d'entraînement non trouvé.")
        print("Exécutez d'abord : python prepare_data.py")
        return

    train_loader = build_dataloader(train_csv, 'train')
    val_loader = build_dataloader(val_csv, 'val')

    # ─── Modèle ───
    model = EmotionDetector().to(device)
    model.count_parameters()

    # ─── Loss ───
    # Charger les poids de classes si disponibles
    weights_path = os.path.join(csv_dir, 'class_weights.json')
    class_weights = None
    if os.path.exists(weights_path):
        with open(weights_path) as f:
            w = json.load(f)
        class_weights = torch.tensor(
            [w.get(cat, 1.0) for cat in EMOTIC_CATEGORIES],
            dtype=torch.float32
        ).to(device)
        print(f"[TRAIN] Poids de classes chargés depuis {weights_path}")

    loss_fn = EmotionLoss(
        vad_weight=cfg.detection.vad_loss_weight,
        use_focal=cfg.detection.use_focal_loss,
        focal_alpha=cfg.detection.focal_alpha,
        focal_gamma=cfg.detection.focal_gamma,
        class_weights=class_weights,
    )

    # ─── Historique ───
    history = {
        'train_loss': [], 'val_loss': [],
        'val_mAP': [], 'val_mae_v': [], 'val_mae_a': [], 'val_mae_d': [],
    }
    best_mAP = 0.0

    # ════════════════════════════════════════════════
    # Phase 1 : Warm-up (backbones gelés)
    # ════════════════════════════════════════════════
    print(f"\n{'='*60}")
    print(f"PHASE 1 : WARM-UP ({cfg.detection.warmup_epochs} époques)")
    print(f"{'='*60}")

    model.freeze_backbones()

    # Seuls les paramètres de la tête et du MLP YOLO sont optimisés
    warmup_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(warmup_params, lr=cfg.detection.lr_head,
                      weight_decay=cfg.detection.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=cfg.detection.warmup_epochs)

    for epoch in range(1, cfg.detection.warmup_epochs + 1):
        t0 = time.time()
        train_losses = train_one_epoch(model, train_loader, loss_fn,
                                       optimizer, device, epoch)
        scheduler.step()

        # Validation
        val_metrics = evaluate_model(model, val_loader, loss_fn, device)
        elapsed = time.time() - t0

        history['train_loss'].append(train_losses['total'])
        history['val_loss'].append(val_metrics['loss_total'])
        history['val_mAP'].append(val_metrics['mAP'])

        print(f"\n  Epoch {epoch}/{cfg.detection.warmup_epochs} "
              f"({elapsed:.0f}s) — "
              f"train_loss={train_losses['total']:.4f} | "
              f"val_loss={val_metrics['loss_total']:.4f} | "
              f"val_mAP={val_metrics['mAP']:.4f} | "
              f"val_MAE_VAD=[{val_metrics['mae_v']:.3f}, "
              f"{val_metrics['mae_a']:.3f}, {val_metrics['mae_d']:.3f}]")

        if val_metrics['mAP'] > best_mAP:
            best_mAP = val_metrics['mAP']
            ckpt_path = os.path.join(cfg.paths.checkpoint_dir, 'best_model.pt')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'mAP': best_mAP,
                'config': cfg.detection.__dict__,
            }, ckpt_path)
            print(f"  ✓ Meilleur modèle sauvegardé (mAP={best_mAP:.4f})")

    # ════════════════════════════════════════════════
    # Phase 2 : Fine-tuning complet
    # ════════════════════════════════════════════════
    remaining_epochs = cfg.detection.num_epochs - cfg.detection.warmup_epochs
    print(f"\n{'='*60}")
    print(f"PHASE 2 : FINE-TUNING ({remaining_epochs} époques)")
    print(f"{'='*60}")

    model.unfreeze_backbones()

    # Lr différencié : backbones à lr réduit
    param_groups = model.get_param_groups(
        lr_head=cfg.detection.lr_head,
        lr_backbone=cfg.detection.lr_backbone,
    )
    optimizer = AdamW(param_groups, weight_decay=cfg.detection.weight_decay)
    scheduler = CosineAnnealingLR(optimizer, T_max=remaining_epochs)

    for epoch_offset in range(1, remaining_epochs + 1):
        epoch = cfg.detection.warmup_epochs + epoch_offset
        t0 = time.time()
        train_losses = train_one_epoch(model, train_loader, loss_fn,
                                       optimizer, device, epoch)
        scheduler.step()

        val_metrics = evaluate_model(model, val_loader, loss_fn, device)
        elapsed = time.time() - t0

        history['train_loss'].append(train_losses['total'])
        history['val_loss'].append(val_metrics['loss_total'])
        history['val_mAP'].append(val_metrics['mAP'])
        history['val_mae_v'].append(val_metrics['mae_v'])
        history['val_mae_a'].append(val_metrics['mae_a'])
        history['val_mae_d'].append(val_metrics['mae_d'])

        print(f"\n  Epoch {epoch}/{cfg.detection.num_epochs} "
              f"({elapsed:.0f}s) — "
              f"train_loss={train_losses['total']:.4f} | "
              f"val_loss={val_metrics['loss_total']:.4f} | "
              f"val_mAP={val_metrics['mAP']:.4f} | "
              f"val_MAE_VAD=[{val_metrics['mae_v']:.3f}, "
              f"{val_metrics['mae_a']:.3f}, {val_metrics['mae_d']:.3f}]")

        if val_metrics['mAP'] > best_mAP:
            best_mAP = val_metrics['mAP']
            ckpt_path = os.path.join(cfg.paths.checkpoint_dir, 'best_model.pt')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'mAP': best_mAP,
                'config': cfg.detection.__dict__,
            }, ckpt_path)
            print(f"  ✓ Meilleur modèle sauvegardé (mAP={best_mAP:.4f})")

    # ─── Sauvegarde finale ───
    final_path = os.path.join(cfg.paths.checkpoint_dir, 'final_model.pt')
    torch.save({
        'epoch': cfg.detection.num_epochs,
        'model_state_dict': model.state_dict(),
        'mAP': val_metrics['mAP'],
        'history': history,
    }, final_path)

    # Sauvegarder l'historique
    hist_path = os.path.join(cfg.paths.log_dir, 'training_history.json')
    with open(hist_path, 'w') as f:
        json.dump(history, f, indent=2)

    print(f"\n{'='*60}")
    print(f"ENTRAÎNEMENT TERMINÉ")
    print(f"  Meilleur mAP : {best_mAP:.4f}")
    print(f"  Checkpoint   : {cfg.paths.checkpoint_dir}")
    print(f"  Historique   : {hist_path}")
    print(f"{'='*60}")




In [ ]:
# ════════════════════════════════════════
# === LANCER L'ENTRAÎNEMENT ===
# ════════════════════════════════════════
# ⏱ Temps estimé : ~2-3h sur T4

train()

---
## Section 5 — Évaluation sur le test set

Charge le meilleur checkpoint et évalue sur le split test :
- **mAP** sur les 26 catégories (multi-label)
- **MAE** et **Pearson r** sur Valence, Arousal, Dominance

In [ ]:
def run_test_evaluation():
    device = torch.device(cfg.device if torch.cuda.is_available() else 'cpu')

    ckpt_path = os.path.join(cfg.paths.checkpoint_dir, 'best_model.pt')
    if not os.path.exists(ckpt_path):
        print("[ERREUR] Aucun checkpoint trouvé.")
        return None

    model = EmotionDetector().to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"[EVAL] Modèle chargé (epoch {ckpt['epoch']}, mAP={ckpt['mAP']:.4f})")

    test_csv = os.path.join(cfg.paths.project_root, 'csv_annotations', 'test.csv')
    test_loader = build_dataloader(test_csv, 'test')

    loss_fn = EmotionLoss(
        vad_weight=cfg.detection.vad_loss_weight,
        use_focal=cfg.detection.use_focal_loss,
    )

    results = evaluate_model(model, test_loader, loss_fn, device)
    print_full_evaluation(results)

    results_path = os.path.join(cfg.paths.log_dir, 'test_results.json')
    json_results = {k: v for k, v in results.items() if not isinstance(v, dict)}
    json_results['ap_per_class'] = results['ap_per_class']
    with open(results_path, 'w') as f:
        json.dump(json_results, f, indent=2)

    return results

test_results = run_test_evaluation()

In [ ]:
# === Visualisation AP par catégorie ===
if test_results:
    ap = test_results['ap_per_class']
    cats = sorted(ap.keys(), key=lambda x: ap[x])
    vals = [ap[c] for c in cats]

    fig, ax = plt.subplots(figsize=(10, 8))
    colors = ['#55A868' if v > 0.3 else '#C44E52' if v < 0.15 else '#CCB974' for v in vals]
    ax.barh(range(len(cats)), vals, color=colors, edgecolor='white')
    ax.set_yticks(range(len(cats)))
    ax.set_yticklabels(cats, fontsize=9)
    ax.set_xlabel('Average Precision', fontsize=12)
    mAP = test_results['mAP']
    ax.axvline(x=mAP, color='navy', linestyle='--', alpha=0.6,
               label=f'mAP = {mAP:.3f}')
    ax.set_title(f'AP par catégorie (mAP = {mAP:.3f})', fontsize=14, fontweight='bold')
    ax.legend()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.paths.log_dir, 'ap_per_category.png'), dpi=150)
    plt.show()

---
## Section 6 — Composant 3 : Fine-tuning LoRA de Stable Diffusion

**Approche :** LoRA (rank 8) sur le UNet de Stable Diffusion v1.5, conditionné sur le vecteur VAD via des prompts textuels structurés.

| Paramètre | Valeur |
|-----------|--------|
| Modèle de base | `runwayml/stable-diffusion-v1-5` |
| LoRA rank | 8 (~1-4M params additionnels) |
| Images d'entraînement | ~5000 (sous-ensemble EMOTIC) |
| Batch size | 1 (+ gradient accumulation ×4) |
| Époques | 10 |

**Temps estimé : ~4-6h sur T4**

In [ ]:
"""
Composant 3 — Génération conditionnelle d'émotions.

Fine-tuning LoRA de Stable Diffusion v1.5 conditionné sur les
vecteurs VAD pour modifier le registre émotionnel d'une scène.

Usage :
    python generation/train_generation.py       # Fine-tuning
    python generation/generate.py               # Génération
    python generation/eval_generation.py        # Évaluation (FID + reclassification)

Temps estimé : ~4-6h de fine-tuning sur T4
"""

import json
import random
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path



# ─────────────────────────────────────────────
# Mapping VAD → prompts textuels
# ─────────────────────────────────────────────

def vad_to_prompt(valence: float, arousal: float, dominance: float) -> str:
    """
    Convertit un vecteur VAD normalisé [0, 1] en prompt textuel
    pour Stable Diffusion.
    """
    def pick_descriptor(value, descriptors):
        if value > 0.66:
            return random.choice(descriptors['high'])
        elif value > 0.33:
            return random.choice(descriptors['mid'])
        else:
            return random.choice(descriptors['low'])

    v_desc = pick_descriptor(valence, VAD_DESCRIPTORS['valence'])
    a_desc = pick_descriptor(arousal, VAD_DESCRIPTORS['arousal'])
    d_desc = pick_descriptor(dominance, VAD_DESCRIPTORS['dominance'])

    prompt = (f"a photograph of a scene with a {v_desc} mood, "
              f"{a_desc} atmosphere, the person appears {d_desc}, "
              f"realistic photography, natural lighting")
    return prompt


def vad_to_prompt_deterministic(valence: float, arousal: float,
                                 dominance: float) -> str:
    """Version déterministe pour l'évaluation (pas de random)."""
    def pick_descriptor(value, descriptors):
        if value > 0.66:
            return descriptors['high'][0]
        elif value > 0.33:
            return descriptors['mid'][0]
        else:
            return descriptors['low'][0]

    v_desc = pick_descriptor(valence, VAD_DESCRIPTORS['valence'])
    a_desc = pick_descriptor(arousal, VAD_DESCRIPTORS['arousal'])
    d_desc = pick_descriptor(dominance, VAD_DESCRIPTORS['dominance'])

    return (f"a photograph of a scene with a {v_desc} mood, "
            f"{a_desc} atmosphere, the person appears {d_desc}, "
            f"realistic photography, natural lighting")


# ─────────────────────────────────────────────
# Dataset pour le fine-tuning
# ─────────────────────────────────────────────

class EMOTICGenerationDataset(Dataset):
    """
    Dataset pour le fine-tuning LoRA de Stable Diffusion.
    Retourne des paires (image, prompt basé sur VAD).
    """
    def __init__(self, csv_path: str, max_samples: int = 5000,
                 image_size: int = 512):
        import pandas as pd
        df = pd.read_csv(csv_path)

        # Filtrer les samples avec VAD valide
        df = df[(df['valence'] >= 0) & (df['arousal'] >= 0) & (df['dominance'] >= 0)]

        # Sous-échantillonnage
        if len(df) > max_samples:
            df = df.sample(n=max_samples, random_state=cfg.seed)

        self.df = df.reset_index(drop=True)
        self.image_size = image_size
        self.images_root = cfg.paths.emotic_images_root

        print(f"[GenDataset] {len(self.df)} samples avec VAD valide")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        folder = str(row['image_folder'])
        filename = str(row['image_file'])

        # Charger l'image
        img_path = os.path.join(self.images_root, folder, filename)
        if not os.path.exists(img_path):
            for sub in ['mscoco/images', 'emotic/images', 'framesdb']:
                alt = os.path.join(self.images_root, sub, filename)
                if os.path.exists(alt):
                    img_path = alt
                    break

        try:
            image = Image.open(img_path).convert('RGB')
            image = image.resize((self.image_size, self.image_size),
                                 Image.LANCZOS)
        except Exception:
            image = Image.new('RGB', (self.image_size, self.image_size))

        # Générer le prompt à partir du VAD
        v = float(row['valence']) / 10.0
        a = float(row['arousal']) / 10.0
        d = float(row['dominance']) / 10.0
        prompt = vad_to_prompt(v, a, d)

        return {
            'image': image,
            'prompt': prompt,
            'vad': [v, a, d],
            'filename': filename,
        }


# ─────────────────────────────────────────────
# Fine-tuning LoRA
# ─────────────────────────────────────────────

def train_lora():
    """
    Fine-tune Stable Diffusion v1.5 avec LoRA sur le dataset EMOTIC.

    Utilise HuggingFace diffusers + peft.
    """
    try:
        from diffusers import StableDiffusionPipeline, DDPMScheduler
        from diffusers import AutoencoderKL, UNet2DConditionModel
        from transformers import CLIPTextModel, CLIPTokenizer
        from peft import LoraConfig, get_peft_model
    except ImportError as e:
        print(f"[ERREUR] Librairies manquantes : {e}")
        print("Installez avec : pip install diffusers transformers peft accelerate")
        return

    device = torch.device(cfg.device if torch.cuda.is_available() else 'cpu')
    gen_cfg = cfg.generation

    print(f"\n{'='*60}")
    print(f"COMPOSANT 3 — FINE-TUNING LoRA")
    print(f"{'='*60}")

    # ─── Charger les composants de SD ───
    print("[GEN] Chargement de Stable Diffusion v1.5...")
    tokenizer = CLIPTokenizer.from_pretrained(
        gen_cfg.base_model, subfolder='tokenizer'
    )
    text_encoder = CLIPTextModel.from_pretrained(
        gen_cfg.base_model, subfolder='text_encoder'
    ).to(device)
    vae = AutoencoderKL.from_pretrained(
        gen_cfg.base_model, subfolder='vae'
    ).to(device)
    unet = UNet2DConditionModel.from_pretrained(
        gen_cfg.base_model, subfolder='unet'
    ).to(device)
    noise_scheduler = DDPMScheduler.from_pretrained(
        gen_cfg.base_model, subfolder='scheduler'
    )

    # Geler tout sauf UNet
    text_encoder.requires_grad_(False)
    vae.requires_grad_(False)

    # ─── Appliquer LoRA au UNet ───
    lora_config = LoraConfig(
        r=gen_cfg.lora_rank,
        lora_alpha=gen_cfg.lora_alpha,
        target_modules=gen_cfg.lora_target_modules,
        lora_dropout=gen_cfg.lora_dropout,
    )
    unet = get_peft_model(unet, lora_config)
    unet.print_trainable_parameters()

    # ─── Dataset ───
    csv_path = os.path.join(cfg.paths.project_root, 'csv_annotations', 'train.csv')
    dataset = EMOTICGenerationDataset(
        csv_path, max_samples=gen_cfg.num_train_images,
        image_size=gen_cfg.image_size
    )

    # ─── Optimiseur ───
    trainable_params = [p for p in unet.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable_params, lr=gen_cfg.lr)

    # ─── Boucle d'entraînement ───
    from torchvision import transforms
    to_tensor = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize([0.5], [0.5]),
    ])

    global_step = 0
    unet.train()

    for epoch in range(1, gen_cfg.num_epochs + 1):
        epoch_loss = 0.0
        num_steps = 0

        for i in range(0, len(dataset), gen_cfg.batch_size):
            sample = dataset[i]
            image = sample['image']
            prompt = sample['prompt']

            # Encoder l'image en latents
            pixel_values = to_tensor(image).unsqueeze(0).to(device)
            with torch.no_grad():
                latents = vae.encode(pixel_values).latent_dist.sample()
                latents = latents * vae.config.scaling_factor

            # Encoder le texte
            text_inputs = tokenizer(
                prompt, padding='max_length',
                max_length=tokenizer.model_max_length,
                truncation=True, return_tensors='pt'
            ).to(device)
            with torch.no_grad():
                encoder_hidden_states = text_encoder(
                    text_inputs.input_ids
                )[0]

            # Ajouter du bruit
            noise = torch.randn_like(latents)
            timesteps = torch.randint(
                0, noise_scheduler.config.num_train_timesteps,
                (latents.shape[0],), device=device
            ).long()
            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

            # Prédire le bruit
            noise_pred = unet(
                noisy_latents, timesteps, encoder_hidden_states
            ).sample

            # Loss
            loss = torch.nn.functional.mse_loss(noise_pred, noise)

            # Gradient accumulation
            loss = loss / gen_cfg.gradient_accumulation_steps
            loss.backward()

            if (num_steps + 1) % gen_cfg.gradient_accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
                optimizer.step()
                optimizer.zero_grad()
                global_step += 1

            epoch_loss += loss.item() * gen_cfg.gradient_accumulation_steps
            num_steps += 1

            if num_steps % 100 == 0:
                avg_loss = epoch_loss / num_steps
                print(f"  Epoch {epoch} [{num_steps}/{len(dataset)}] "
                      f"loss={avg_loss:.6f}")

        avg_loss = epoch_loss / max(num_steps, 1)
        print(f"\n  Epoch {epoch}/{gen_cfg.num_epochs} — "
              f"loss={avg_loss:.6f} — steps={global_step}")

    # ─── Sauvegarde des poids LoRA ───
    lora_path = os.path.join(cfg.paths.checkpoint_dir, 'lora_weights')
    unet.save_pretrained(lora_path)
    print(f"\n[GEN] Poids LoRA sauvegardés dans {lora_path}")

    # Sauvegarder aussi la config
    config_path = os.path.join(lora_path, 'gen_config.json')
    with open(config_path, 'w') as f:
        json.dump(gen_cfg.__dict__, f, indent=2, default=str)

    print("[GEN] Fine-tuning terminé.")




In [ ]:
# ════════════════════════════════════════
# === LANCER LE FINE-TUNING LoRA ===
# ════════════════════════════════════════
# ⏱ Temps estimé : ~4-6h sur T4

train_lora()

---
## Section 7 — Génération d'images & Évaluation inter-tâches

1. **Génération** : N images conditionnées sur des vecteurs VAD variés
2. **FID** : qualité visuelle comparée aux images réelles EMOTIC
3. **Reclassification** : les images générées sont passées dans le Composant 2 pour mesurer la cohérence émotion cible ↔ émotion prédite

In [ ]:
# (fonctions vad_to_prompt_deterministic et classes déjà définies ci-dessus)
"""
Génération d'images et évaluation du Composant 3.

Deux scripts en un :
  1. generate_images() : génère des images conditionnées sur VAD
  2. evaluate_generation() : calcule FID + reclassification score
"""

import json
import torch
import numpy as np
from PIL import Image
from pathlib import Path



def generate_images(num_images: int = None, vad_targets: list = None):
    """
    Génère des images avec le modèle SD fine-tuné LoRA.

    Args:
        num_images: nombre d'images à générer (défaut: cfg.generation.num_eval_images)
        vad_targets: liste de tuples (v, a, d) — si None, échantillonne aléatoirement
    """
    from diffusers import StableDiffusionPipeline
    from peft import PeftModel

    gen_cfg = cfg.generation
    device = torch.device(cfg.device if torch.cuda.is_available() else 'cpu')
    num_images = num_images or gen_cfg.num_eval_images

    print(f"\n{'='*60}")
    print(f"GÉNÉRATION DE {num_images} IMAGES")
    print(f"{'='*60}")

    # ─── Charger le pipeline ───
    print("[GEN] Chargement du pipeline SD + LoRA...")
    pipe = StableDiffusionPipeline.from_pretrained(
        gen_cfg.base_model,
        torch_dtype=torch.float16 if device.type == 'cuda' else torch.float32,
        safety_checker=None,
    ).to(device)

    # Charger les poids LoRA
    lora_path = os.path.join(cfg.paths.checkpoint_dir, 'lora_weights')
    if os.path.exists(lora_path):
        pipe.unet = PeftModel.from_pretrained(pipe.unet, lora_path)
        print(f"  Poids LoRA chargés depuis {lora_path}")
    else:
        print(f"  [WARN] Pas de poids LoRA trouvés, utilisation du modèle de base")

    # ─── Générer les cibles VAD ───
    if vad_targets is None:
        # Échantillonner des VAD variés pour couvrir l'espace
        vad_targets = []
        np.random.seed(cfg.seed)
        for _ in range(num_images):
            v = np.random.uniform(0, 1)
            a = np.random.uniform(0, 1)
            d = np.random.uniform(0, 1)
            vad_targets.append((v, a, d))

    # ─── Génération ───
    output_dir = cfg.paths.generation_output_dir
    os.makedirs(output_dir, exist_ok=True)


    metadata = []

    for i, (v, a, d) in enumerate(vad_targets[:num_images]):
        prompt = vad_to_prompt_deterministic(v, a, d)

        image = pipe(
            prompt,
            num_inference_steps=gen_cfg.num_inference_steps,
            guidance_scale=gen_cfg.guidance_scale,
            generator=torch.Generator(device=device).manual_seed(cfg.seed + i),
        ).images[0]

        # Sauvegarder
        img_path = os.path.join(output_dir, f'gen_{i:04d}.png')
        image.save(img_path)

        metadata.append({
            'index': i,
            'filename': f'gen_{i:04d}.png',
            'prompt': prompt,
            'vad_target': {'valence': v, 'arousal': a, 'dominance': d},
        })

        if (i + 1) % 50 == 0:
            print(f"  [{i+1}/{num_images}] générées")

    # Sauvegarder les métadonnées
    meta_path = os.path.join(output_dir, 'generation_metadata.json')
    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f"\n[GEN] {len(metadata)} images sauvegardées dans {output_dir}")
    return metadata


def compute_fid(real_dir: str, gen_dir: str):
    """
    Calcule le FID entre images réelles et générées.
    Utilise clean-fid ou torch-fidelity.
    """
    try:
        from cleanfid import fid
        score = fid.compute_fid(real_dir, gen_dir)
        print(f"  FID (clean-fid) : {score:.2f}")
        return score
    except ImportError:
        pass

    try:
        from torch_fidelity import calculate_metrics
        metrics = calculate_metrics(
            input1=gen_dir,
            input2=real_dir,
            fid=True,
            verbose=False,
        )
        score = metrics['frechet_inception_distance']
        print(f"  FID (torch-fidelity) : {score:.2f}")
        return score
    except ImportError:
        print("  [WARN] Ni clean-fid ni torch-fidelity installé.")
        print("  Installez avec : pip install clean-fid")
        return -1.0


def reclassification_score():
    """
    Feedback loop inter-tâches :
    Passe les images générées dans le Composant 2 et mesure
    la cohérence entre le VAD cible et le VAD prédit.
    """
    from models.emotion_detector import EmotionDetector
    import torchvision.transforms as T

    device = torch.device(cfg.device if torch.cuda.is_available() else 'cpu')

    # Charger le classificateur
    ckpt_path = os.path.join(cfg.paths.checkpoint_dir, 'best_model.pt')
    if not os.path.exists(ckpt_path):
        print("  [ERREUR] Pas de modèle de détection entraîné.")
        return {}

    model = EmotionDetector().to(device)
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    # Charger les métadonnées de génération
    meta_path = os.path.join(cfg.paths.generation_output_dir,
                              'generation_metadata.json')
    if not os.path.exists(meta_path):
        print("  [ERREUR] Pas de métadonnées de génération.")
        return {}

    with open(meta_path) as f:
        metadata = json.load(f)

    # Transforms
    transform = T.Compose([
        T.Resize((224, 224)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                     std=[0.229, 0.224, 0.225]),
    ])

    # Prédictions
    vad_targets = []
    vad_preds = []

    with torch.no_grad():
        for entry in metadata:
            img_path = os.path.join(cfg.paths.generation_output_dir,
                                     entry['filename'])
            try:
                img = Image.open(img_path).convert('RGB')
            except Exception:
                continue

            img_tensor = transform(img).unsqueeze(0).to(device)

            # Pour la reclassification, on utilise l'image comme
            # person crop ET contexte (scène générée = scène complète)
            yolo_feat = torch.zeros(1, 80, device=device)

            _, vad_pred = model(img_tensor, img_tensor, yolo_feat)

            vad_target = entry['vad_target']
            vad_targets.append([vad_target['valence'],
                                vad_target['arousal'],
                                vad_target['dominance']])
            vad_preds.append(vad_pred.cpu().numpy()[0].tolist())

    vad_targets = np.array(vad_targets)
    vad_preds = np.array(vad_preds)

    # Métriques
    from scipy.stats import pearsonr
    results = {}
    for i, name in enumerate(['valence', 'arousal', 'dominance']):
        mae = np.mean(np.abs(vad_targets[:, i] - vad_preds[:, i]))
        r, p = pearsonr(vad_targets[:, i], vad_preds[:, i])
        results[f'{name}_mae'] = float(mae)
        results[f'{name}_pearson_r'] = float(r)
        results[f'{name}_pearson_p'] = float(p)
        print(f"  {name:10s} — MAE={mae:.4f}, Pearson r={r:.4f} (p={p:.4f})")

    return results


def evaluate_generation():
    """Évaluation complète du Composant 3."""
    print(f"\n{'='*60}")
    print(f"ÉVALUATION DE LA GÉNÉRATION")
    print(f"{'='*60}")

    gen_dir = cfg.paths.generation_output_dir

    # 1. FID
    print("\n[1] Fréchet Inception Distance (FID)")
    # Créer un dossier d'images réelles pour comparaison
    # (idéalement un subset du val/test EMOTIC)
    real_dir = os.path.join(cfg.paths.project_root, 'real_subset')
    if os.path.exists(real_dir) and len(os.listdir(real_dir)) > 0:
        fid_score = compute_fid(real_dir, gen_dir)
    else:
        print("  [INFO] Créez un dossier 'real_subset' avec ~500 images EMOTIC")
        print("  pour calculer le FID. Voir le notebook pour le script.")
        fid_score = -1.0

    # 2. Reclassification
    print("\n[2] Reclassification par le Composant 2")
    reclass_results = reclassification_score()

    # Résumé
    all_results = {
        'fid': fid_score,
        **reclass_results,
    }
    results_path = os.path.join(cfg.paths.log_dir, 'generation_results.json')
    with open(results_path, 'w') as f:
        json.dump(all_results, f, indent=2)

    print(f"\n[EVAL] Résultats sauvegardés dans {results_path}")
    return all_results




In [ ]:
# === Préparer un subset d'images réelles pour le FID ===
# Copier ~500 images EMOTIC dans un dossier dédié
import shutil
real_subset_dir = os.path.join(cfg.paths.project_root, 'real_subset')
os.makedirs(real_subset_dir, exist_ok=True)

if len(os.listdir(real_subset_dir)) == 0:
    val_csv = os.path.join(cfg.paths.project_root, 'csv_annotations', 'val.csv')
    if os.path.exists(val_csv):
        df_val = pd.read_csv(val_csv)
        # Prendre 500 images uniques
        unique_imgs = df_val[['image_folder', 'image_file']].drop_duplicates().head(500)
        copied = 0
        for _, row in unique_imgs.iterrows():
            src = os.path.join(cfg.paths.emotic_images_root, str(row['image_folder']), str(row['image_file']))
            if not os.path.exists(src):
                for sub in ['mscoco/images', 'emotic/images', 'framesdb']:
                    alt = os.path.join(cfg.paths.emotic_images_root, sub, str(row['image_file']))
                    if os.path.exists(alt):
                        src = alt
                        break
            if os.path.exists(src):
                dst = os.path.join(real_subset_dir, str(row['image_file']))
                if not os.path.exists(dst):
                    try:
                        # Redimensionner à 512x512 pour le FID
                        img = Image.open(src).convert('RGB').resize((512, 512), Image.LANCZOS)
                        img.save(dst)
                        copied += 1
                    except: pass
        print(f"✅ {copied} images réelles copiées dans {real_subset_dir}")

In [ ]:
# === Générer des images ===
# Réduire num_images pour un test rapide
metadata = generate_images(num_images=100)

In [ ]:
# === Évaluation complète ===
gen_results = evaluate_generation()

In [ ]:
# === Afficher des exemples générés ===
gen_dir = cfg.paths.generation_output_dir
meta_path = os.path.join(gen_dir, 'generation_metadata.json')

if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)

    fig, axes = plt.subplots(2, 5, figsize=(20, 8))
    for i, ax in enumerate(axes.flat):
        if i < len(meta):
            img_path = os.path.join(gen_dir, meta[i]['filename'])
            if os.path.exists(img_path):
                img = plt.imread(img_path)
                vad = meta[i]['vad_target']
                ax.imshow(img)
                ax.set_title(f"V={vad['valence']:.2f} A={vad['arousal']:.2f}\nD={vad['dominance']:.2f}", fontsize=8)
            ax.axis('off')
    plt.suptitle('Exemples de scènes générées (conditionnées sur VAD)', fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(cfg.paths.log_dir, 'generated_examples.png'), dpi=150)
    plt.show()

---
## Section 8 — Visualisations pour le poster

Génère toutes les figures nécessaires en haute résolution (300 DPI) :
1. Distribution des classes EMOTIC
2. Courbes d'entraînement (loss + mAP)
3. AP par catégorie
4. Exemples de génération

In [ ]:
def create_poster_figures():
    log_dir = cfg.paths.log_dir
    figs_dir = os.path.join(log_dir, 'poster_figures')
    os.makedirs(figs_dir, exist_ok=True)

    # --- Figure 1 : Distribution des classes ---
    train_csv = os.path.join(cfg.paths.project_root, 'csv_annotations', 'train.csv')
    if os.path.exists(train_csv):
        df = pd.read_csv(train_csv)
        counts = df[EMOTIC_CATEGORIES].sum().sort_values(ascending=True)
        fig, ax = plt.subplots(figsize=(8, 7))
        counts.plot(kind='barh', ax=ax, color='#4C72B0', edgecolor='white')
        ax.set_xlabel("Nombre d'annotations", fontsize=12)
        ax.set_title('Distribution des catégories EMOTIC', fontsize=14, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.savefig(os.path.join(figs_dir, 'fig1_class_distribution.png'), dpi=300, bbox_inches='tight')
        plt.show()
        print("  ✓ Figure 1")

    # --- Figure 2 : Courbes d'entraînement ---
    hist_path = os.path.join(log_dir, 'training_history.json')
    if os.path.exists(hist_path):
        with open(hist_path) as f:
            h = json.load(f)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
        epochs = range(1, len(h['train_loss']) + 1)
        ax1.plot(epochs, h['train_loss'], '-o', color='#4C72B0', markersize=3, label='Train')
        ax1.plot(epochs, h['val_loss'], '-o', color='#DD8452', markersize=3, label='Validation')
        ax1.axvline(x=cfg.detection.warmup_epochs, color='gray', linestyle='--', alpha=0.5)
        ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss'); ax1.legend()
        ax1.set_title('Loss combinée (BCE + λ·MSE)')
        ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)
        ax2.plot(epochs, h['val_mAP'], '-o', color='#55A868', markersize=3)
        ax2.set_xlabel('Epoch'); ax2.set_ylabel('mAP')
        ax2.set_title('mean Average Precision (val)')
        ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)
        plt.tight_layout()
        plt.savefig(os.path.join(figs_dir, 'fig2_training_curves.png'), dpi=300, bbox_inches='tight')
        plt.show()
        print("  ✓ Figure 2")

    # --- Figure 3 : AP par catégorie ---
    results_path = os.path.join(log_dir, 'test_results.json')
    if os.path.exists(results_path):
        with open(results_path) as f:
            results = json.load(f)
        ap = results.get('ap_per_class', {})
        if ap:
            cats = sorted(ap.keys(), key=lambda x: ap[x])
            vals = [ap[c] for c in cats]
            fig, ax = plt.subplots(figsize=(8, 7))
            colors = ['#55A868' if v > 0.3 else '#C44E52' if v < 0.15 else '#CCB974' for v in vals]
            ax.barh(range(len(cats)), vals, color=colors, edgecolor='white')
            ax.set_yticks(range(len(cats)))
            ax.set_yticklabels(cats, fontsize=9)
            ax.set_xlabel('Average Precision', fontsize=12)
            mAP = results.get('mAP', 0)
            ax.axvline(x=mAP, color='navy', linestyle='--', alpha=0.6)
            ax.set_title(f'AP par catégorie (mAP = {mAP:.3f})', fontsize=14, fontweight='bold')
            ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
            plt.tight_layout()
            plt.savefig(os.path.join(figs_dir, 'fig3_ap_categories.png'), dpi=300, bbox_inches='tight')
            plt.show()
            print("  ✓ Figure 3")

    print(f"\n✅ Figures poster sauvegardées dans {figs_dir}")

create_poster_figures()

---
## 🎉 Pipeline terminée

Tous les résultats sont dans `/content/emotic_project/` :

| Dossier | Contenu |
|---------|---------|
| `checkpoints/` | Meilleur modèle + poids LoRA |
| `logs/` | Historique d'entraînement, résultats test, métriques génération |
| `logs/poster_figures/` | Figures haute résolution pour le poster |
| `generated/` | Images générées + métadonnées |
| `yolo_features/` | Vecteurs YOLO pré-calculés |
| `csv_annotations/` | CSV des splits + poids de classes |